# Job Listing Prioritization for Career Changers: EDA/SDA 3 Lean (Text features)

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

**What do the listing descriptions say?** This notebook turns the description text of the Lean chain's in-scope listings into **features**, and reports the text rules as **keyword rules with their firing rates**, so `EDA_SDA_4` can test every feature (text included) and the app can label what a listing asks and offer opt-in filters. **Decided 2026-10-13: the validation samples (section 5) will not be hand-checked,** so the text blockers (years asked, degree required, clearance, no sponsorship and the other career-change flags) are **unvalidated**. They are labels on each listing and filters a career changer turns on (for example "I have no degree"), never hidden by default. The same decision covers EDA_SDA_2_Lean's two title rules (staffing agency from recruiter titles; the 13-1161 split into analysts and marketing specialists): they are not hand-checked here either, and are labels and opt-in filters. Only reliable rules hide by default (seniority, likely ghosts, occupations that aren't a tech path); every rule reports how many listings it would hide. **It builds no matching models:** résumé-to-listing fit (keyword overlap, TF-IDF, BGE embeddings, RRF) is in `ML_Fit`, which uses this notebook's cleaned text and skills. A text feature is kept only if it adds information beyond the non-text features.

**For career changers,** the most important text features are what a listing asks of a newcomer (experience, education, licenses, training) and which of its skills carry over from other fields.

**Inputs (the Lean chain):** `EDA_SDA_1_Lean`'s in-scope table (`analysis_table.parquet`) and
`EDA_SDA_2_Lean`'s rule flags (`scope_flags.parquet`), joined by `id`, plus the description text for those
listings (`scope_descriptions.parquet`). All three are in `data/clean/lean/`.

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the data, how does tech compare to other fields, and what's in scope? | Scope table, descriptions, occupation summary, company activity |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? | Scope table, non-text features |
| **EDA_SDA_3** | **What do the descriptions say? Skills mapped to O*NET** | **Scope table + text features** |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

**View:** listings with a description (the share 1b reports from `has_desc`). Text features are **unknown** for the rest, never "no", and "has a description" stays a standard confounder: listings without one close much faster.

**Data rules:** aggregates only; never print descriptions, titles or sample rows, except in cells tagged `data-rows` (review cells for a teammate to check locally; cleared before every commit). Text stays local: no listing text goes to outside AI services. Clear outputs before committing.

### Clarifying questions
- What does description length actually capture: employer intent, or boilerplate?
- Does urgency wording, read in context rather than by keyword, relate to how long listings stay open? (EDA_SDA_1_Lean left it here: its keyword flag misfired and showed no gap.)
- Can evergreen wording ("always accepting applications") identify ghost listings?
- Are EDA_SDA_1_Lean's career-change flags (years required, degree required, training and career-changer wording) accurate, and what do they miss? Two are urgent: EDA_SDA_1_Lean's "asks 5+ years" rule would hide a large share of open listings if a user turned it on, and its degree flag fires on very few (both measured here in 3a and 3b, and tagged in EDA_SDA_2_Lean 4.5 as `asks_5_plus_years` and `degree_no_equivalent`).
- Which requirements are labels and opt-in filters (years asked, degree, clearance, sponsorship, certification), and how many listings would each hide if a user turned it on? (Not validated: see the 2026-10-13 decision above.)
- Which skills does each listing ask for, mapped to O*NET skills, so `ML_Fit` can match on skills rather than words?

### Roadmap

1. **Setup:** the scope table and its description text
2. **Description cleanup:** HTML to structured text, boilerplate removal, normalization and lemmas
3. **The text rules:** years of experience (required vs. preferred) and degree requirements, kept as labels and opt-in filters
4. **Other text features:** evergreen wording, hiring-difficulty phrases, licenses and certifications, training and career-changer wording, urgency, remote, length
5. **Validation samples:** written, but not being marked (decided 2026-10-13); rules are reported as keyword rules with their firing rates
6. **Skills mapped to O*NET** (for `ML_Fit`)
7. **Final table:** text rules saved as labels with the count each would hide (nothing hidden by default, nothing removed), for EDA_SDA_4 and `ML_Fit`
8. **Conclusion**


## Plan: text features to build

| Feature | Question | Notes |
|:--|:--|:--|
| **Description cleanup** (moved here from EDA_SDA_2 2b, which EDA_SDA_2_Lean leaves out) | Strip HTML and boilerplate (equal-opportunity statements, benefits, "about us") | `desc_clean` and `desc_words_clean`; the base for every feature below and for `ML_Fit` |
| Description length | Is it employer intent, or just boilerplate? | Measured on `desc_clean` |
| Urgency wording | Does real urgency (in context, not "urgent care") go with shorter or longer windows? | EDA_SDA_1_Lean's keyword flag showed no gap; may mark hard-to-fill roles |
| Remote mentions | Does remote add beyond `contract_types`? | |
| **Evergreen wording** | Listings that collect applications instead of filling one job: a ghost-listing signal | Six phrase groups, each scored and checked on its own: **talent pool** ("join our talent community", "future opportunities"), **no specific opening** ("general application", "not for a specific position"), **always hiring** ("always accepting applications", "rolling basis"), **many openings in one ad** ("multiple openings", "various locations"), **pipeline labels** ("pipeline requisition", "evergreen requisition"), **staffing-style** ("on behalf of our client"); plus a generic-title check ("Software Engineer – All Levels"). Compared with EDA_SDA_2_Lean's likely-ghost flag (3.1): kept only if it adds to "old and from a quiet company" |
| Hiring-difficulty phrases | "Sign-on bonus", "hiring immediately", "relocation assistance", "will train" | Pairs with reposts and listing age |
| Years of experience required | "3+ years": is EDA_SDA_1_Lean's rule accurate? Required or preferred? In the field, or any experience? | The most common blocker for career changers |
| Education required | "Bachelor's required" vs. "or equivalent experience"; field-specific degrees | EDA_SDA_1_Lean's pattern only fires when "required" comes within six words of "bachelor's", so the most common wording ("Bachelor's degree in Computer Science or a related field") never matches, and it flags only a small fraction of listings (3b measures both here). Rebuilt in section 3, with requirements and preferred sections told apart |
| **Clearance, sponsorship, travel, shifts** (added 2026-10-10) | Security clearance required; "no visa sponsorship"; travel percentage; shift or on-call work; contract-to-hire | Clearance and sponsorship are blockers for some career changers: a label on the listing and an opt-in filter in the app, like a missing degree (never hidden by default) |
| Licenses and certifications required | "Security+", "CCNA", "CISSP" and the like | A required credential the career changer lacks: a label and an opt-in filter |
| Training and career-changer wording | "Will train", "paid training", "career changers welcome", "transferable skills" | Reports EDA_SDA_1_Lean's flags as keyword rules with their firing rates (the hand-check sample is not being marked, decided 2026-10-13) |
| **Text-rule filters** | Which years-asked threshold and which required credentials would a user filter on, per role bucket? | Not validated (decided 2026-10-13): saved in this notebook's final table as labels, with a count of the listings each would hide if turned on (opt-in in the app, never hidden by default, never removed) |
| Required vs. preferred qualifications | Split on headings like "Requirements" and "Nice to have" | Part of fit and gap analysis in `ML_Fit` |
| **Skills, mapped to O*NET** | Matched as whole phrases against O*NET's tools and skill names on requirement lines (PredictLeads `tags` are sparse), section 6 | `ML_Fit` matches résumés to listings on these skills, so "transferable" doesn't depend on the old field's vocabulary |

**Moved to `ML_Fit`:** role similarity, TF-IDF and embeddings for matching. **Moved to `ML_Rank`:** TF-IDF terms that predict how fast a listing closes (a text input to the timing model). Topic modeling stays a stretch goal.

## Plan: preprocessing

Course standard (MLforTexts): normalize text (lowercase, regex to keep letters), remove NLTK stopwords, lemmatize with spaCy. Advanced: remove boilerplate where headings make it detectable. Any vocabulary learned from the text is fit on the training months only.

**Evaluation (course standard):** the plan was precision and recall on a hand-checked random sample against EDA_SDA_1_Lean's keyword rule. **Decided 2026-10-13: the samples will not be hand-checked,** so each rule is reported as a keyword rule with its firing rate, against EDA_SDA_1_Lean's rule where one exists, and not as a validated blocker.


## 1. Setup

In [ ]:
# 1a Libraries, paths and the shared code: src/text_features.py does the cleanup and the text rules batch by batch,
# so the full scope's descriptions never sit in memory at once.
import gc
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, INK, MUTED, CLEAN_DIR,
    bar_h, finish, show, show_shaded, use_chart_style,
)
from src import text_features as tf
use_chart_style()

TRAIN_END = "2026-07"  # anything learned from the text (template lines) is learned on April–July only, ML_Rank's training months
# The Lean chain writes to data/clean/lean/, and so does this notebook. The caches point there too, so no file
# left by a run of the original EDA_SDA_3 is ever picked up. They all move to data/clean/ when the Lean
# notebooks replace the originals.
LEAN_DIR = CLEAN_DIR / "lean"
SCOPE_TABLE = LEAN_DIR / "analysis_table.parquet"  # EDA_SDA_1_Lean's in-scope table (was EDA_SDA_2's scope table)
FLAGS_TABLE = LEAN_DIR / "scope_flags.parquet"  # EDA_SDA_2_Lean's rule flags, joined by id
TEXT_TABLE = LEAN_DIR / "text_table.parquet"  # this notebook's final table (7a), for EDA_SDA_4
DESC_PATH = LEAN_DIR / "scope_descriptions.parquet"  # EDA_SDA_1_Lean: id, description (HTML)
FEATURES_PATH = LEAN_DIR / "eda3_text_features.parquet"  # every listing's text features (cache; delete to rebuild)
CLEAN_PATH = LEAN_DIR / "eda3_desc_clean.parquet"  # every listing's cleaned text, for ML_Fit
LEMMA_PATH = LEAN_DIR / "eda3_lemmas_sample.parquet"  # lemmas for a stratified sample (2c)
LANG_PATH = LEAN_DIR / "eda3_language.parquet"  # per-description language scores (2d)
REMOVED_PATH = LEAN_DIR / "eda3_removed_words.json"  # words removed per boilerplate reason (2b)
VALIDATION_DIR = LEAN_DIR / "eda3_validation"  # hand-check samples (section 5); local only, like all of data/
TEMPLATE_SAMPLE = 120_000  # listings used to learn shared template lines (training months, with a description)
LEMMA_SAMPLE = 150_000  # listings lemmatized with spaCy (stratified by role bucket)
_t0 = time.time()

In [ ]:
# 1b Load the columns this notebook needs from the Lean chain: EDA_SDA_1_Lean's in-scope table and
# EDA_SDA_2_Lean's rule flags, joined by id, in place of EDA_SDA_2's single scope table. Each column is taken from
# whichever table has it, and anything the Lean chain no longer carries is reported instead of quietly filled.
# Only counts are shown.
SCOPE_COLS = ["id", "company_id", "first_seen", "month", "role_bucket", "tripleten_program", "has_desc", "live",
              "days_open", "source_type", "is_repost", "career_open", "years_required", "degree_required",
              "equivalent_experience", "entry_wording", "changer_wording", "urgency", "mentions_remote",
              "company_earlier", "onet_code"]                           # EDA_SDA_1_Lean's in-scope table
LEAN_FLAGS = ["id", "level", "cc_visible", "likely_ghost", "open_90_days"]   # EDA_SDA_2_Lean's rule flags
# hidden_occupation has no equivalent on purpose: EDA_SDA_1_Lean drops the three occupations that aren't a tech
# path as rows, so they are absent from the scope rather than flagged in it.
scope_have = set(pq.ParquetFile(SCOPE_TABLE).schema.names)
flags_have = set(pq.ParquetFile(FLAGS_TABLE).schema.names)
jobs = pd.read_parquet(SCOPE_TABLE, columns=[c for c in SCOPE_COLS if c in scope_have])
flags = pd.read_parquet(FLAGS_TABLE, columns=[c for c in LEAN_FLAGS if c in flags_have])
jobs = jobs.merge(flags[[c for c in flags.columns if c == "id" or c not in jobs.columns]], on="id", how="left")
jobs["month"] = jobs.month.astype(str)

# Two of EDA_SDA_2's columns that EDA_SDA_2_Lean does not carry, rebuilt so sections 4b and 4c read as they did
jobs["stale"] = jobs.open_90_days if "open_90_days" in jobs else pd.Series(pd.NA, index=jobs.index, dtype="boolean")
VOL_BANDS = ["1-4", "5-49", "50-499", "500+"]   # EDA_SDA_2 3.3's bands; company_earlier counts only the company's
jobs["volume_band"] = (pd.cut(jobs.company_earlier + 1, [0, 4, 49, 499, np.inf], labels=VOL_BANDS)  # earlier
                       if "company_earlier" in jobs                                                 # listings, so
                       else pd.Categorical([None] * len(jobs), categories=VOL_BANDS))               # it can't leak
meta = jobs.set_index("id")[["company_id", "month"]]
absent = [c for c in SCOPE_COLS + LEAN_FLAGS if c not in scope_have | flags_have]

show("1b The Lean chain's in-scope table and its text", pd.DataFrame({"value": {
    "listings (EDA_SDA_1_Lean's in-scope table)": f"{len(jobs):,}",
    "with EDA_SDA_2_Lean's flags": f"{jobs.level.notna().mean():.1%}" if "level" in jobs else "no flags table",
    "with a description (has_desc)": f"{jobs.has_desc.mean():.1%}",
    "descriptions file": f"{DESC_PATH.name}: {pq.ParquetFile(DESC_PATH).metadata.num_rows:,} rows, {DESC_PATH.stat().st_size / 1e9:.2f} GB",
    "first seen in the training months (April to July)": f"{(jobs.month <= TRAIN_END).sum():,}",
    "volume_band rebuilt from": "company_earlier" if "company_earlier" in jobs else "nothing: it is blank",
}}))
if absent:
    print("not in the Lean chain, so the cells that use them say so and skip:", absent)


## 2. Description cleanup

**Why:** every text feature, and `ML_Fit`'s matching, should read what the listing asks for, not the legal and benefits text most listings share. At full scale the descriptions are several GB of text, so **the cleanup runs batch by batch** (`src/text_features.py`): each batch is cleaned, its features computed and both written to disk, then dropped from memory.

| Step | What | Output |
|:--|:--|:--|
| 2a | HTML to text, one line per paragraph, list item or heading (long unbroken text split into sentences) | Lines per listing |
| 2b | Remove boilerplate line by line: known patterns (equal opportunity and legal, benefits, application logistics), "About the company" sections, and text shared word for word across 5+ employers (learned on a sample of April–July listings; lines with a requirement cue are never cut) | `desc_clean` (`eda3_desc_clean.parquet`), `words_clean` |
| 2c | Course standard: lowercase, letters only, NLTK stopwords, spaCy lemmas, on a stratified sample | `desc_lemmas` (for word counts and TF-IDF later) |
| 2d | Check the text really is English: the scope cut only trusts the provider's `language` label, so each description's English function words are counted against five other languages' | `not_english` (`eda3_language.parquet`) |

The rules in sections 3–4 run on the cleaned text, which keeps case, numbers and punctuation ("3+ years", "B.S.", "Security+"), and tell **requirement lines from preferred ones** using the headings kept in 2a.

In [ ]:
# 2a–2b Learn shared template lines on a sample of training-month listings, then clean every description and compute
# its text features, batch by batch on several processes (cached: delete eda3_text_features.parquet to rebuild).
have = jobs[jobs.has_desc & (jobs.month <= TRAIN_END)]
sample_ids = set(have.sample(min(TEMPLATE_SAMPLE, len(have)), random_state=42).id)
if tf.cache_valid(FEATURES_PATH, DESC_PATH, REMOVED_PATH, CLEAN_PATH):  # an unreadable file (a killed run) means rebuild
    removed = json.loads(REMOVED_PATH.read_text())
    print(f"loaded the text features from {FEATURES_PATH.name} (delete it to rebuild 2a–2b)")
else:
    templates = tf.learn_templates(DESC_PATH, sample_ids, meta.company_id)  # streams batches on several processes; keeps only line keys
    print(f"shared template lines learned on {len(sample_ids):,} April–July listings: {len(templates):,}")
    start = time.time()
    removed = tf.build(DESC_PATH, meta, templates, FEATURES_PATH, CLEAN_PATH,
                       progress=lambda n, k: print(f"  batch {n + 1}: {k:,} listings, {time.time() - start:.0f}s", end="\r"))
    REMOVED_PATH.write_text(json.dumps(removed))
    print(f"\ncleaned and featurized every description in {(time.time() - start) / 60:.1f} min")
text = pd.read_parquet(FEATURES_PATH, dtype_backend="numpy_nullable")  # blanks stay unknown (NA), not None in object columns
jobs = jobs.merge(text, on="id", how="left", suffixes=("_eda1", ""))
has_text = jobs.words_raw.fillna(0) > 0
by_reason = pd.Series(removed, dtype=float).sort_values(ascending=False)
show("2b Words removed as boilerplate (listings with a description)", pd.DataFrame({
    "words removed": by_reason,
    "share of all description words": by_reason / jobs.words_raw.sum(),
}).round(3))
src = jobs[has_text].groupby("source_type")
show("2b Share of each description kept after cleanup, by source (median per listing)", pd.DataFrame({
    "median words before": src.words_raw.median(), "median words after": src.words_clean.median(),
    "median share kept": src.apply(lambda g: (g.words_clean / g.words_raw).median(), include_groups=False),
    "cleaned to nothing": src.apply(lambda g: g.words_clean.eq(0).mean(), include_groups=False),
    "listings": src.size(),
}).round(3))

**What the data shows**

_(to write from this notebook's own run on the Lean chain. The original's numbers are gone because its scope was EDA_SDA_2's 908,291 listings and this one is EDA_SDA_1_Lean's 878,937, so not one of them carries across. This cell answers: how many listings the cleanup ran on and how many have a description, what boilerplate was removed by kind, and how much of its text a listing keeps.)_


In [ ]:
# 2b review: a few random listings after cleanup, for a teammate to check locally that 2b keeps the requirements.
# Shows real listing text. Tagged "data-rows": clear this output before committing.
clean_sample = pq.ParquetFile(CLEAN_PATH).read_row_group(0).to_pandas().sample(3, random_state=7)
for t in clean_sample.desc_clean:
    print("=" * 100, "\n", t[:2500])

In [ ]:
# 2c Normalize and lemmatize (course standard: lowercase, letters only, NLTK stopwords, spaCy lemmas) on a stratified
# sample, since spaCy on every listing would take hours. desc_lemmas feeds word counts and TF-IDF in ML_Fit and ML_Rank.
import nltk
import spacy
try:
    STOP = set(nltk.corpus.stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)  # one-time download of NLTK's word list (no project data leaves the laptop)
    STOP = set(nltk.corpus.stopwords.words("english"))

if LEMMA_PATH.exists():
    lem = pd.read_parquet(LEMMA_PATH)
    print(f"loaded {len(lem):,} lemmatized listings from {LEMMA_PATH.name}")
else:
    pool = jobs[has_text]
    lem_ids = set(pool.groupby("role_bucket").sample(frac=min(LEMMA_SAMPLE / len(pool), 1), random_state=42).id)
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # the tagger and lemmatizer are all that's needed
    parts = []
    for batch in pq.ParquetFile(CLEAN_PATH).iter_batches(batch_size=20_000):
        d = batch.to_pandas()
        d = d[d.id.isin(lem_ids)]
        lemmas = [" ".join(t.lemma_.lower() for t in doc if t.is_alpha and t.lemma_.lower() not in STOP)
                  for doc in nlp.pipe(d.desc_clean.str.replace("\n", ". ").str[:20000], batch_size=64)]
        parts.append(pd.DataFrame({"id": d.id.to_numpy(), "desc_lemmas": lemmas}))
    lem = pd.concat(parts, ignore_index=True)
    lem.to_parquet(LEMMA_PATH, index=False)
    print(f"saved {len(lem):,} lemmatized listings to {LEMMA_PATH.name}")
words_lem = lem.desc_lemmas.str.split().str.len()
show("2c Words per description at each step (listings with a description)", pd.DataFrame({
    "after HTML removal": jobs.words_raw[has_text].describe(percentiles=[0.1, 0.5, 0.9]),
    "after boilerplate removal": jobs.words_clean[has_text].describe(percentiles=[0.1, 0.5, 0.9]),
    "lemmas, stopwords removed (sample)": words_lem.describe(percentiles=[0.1, 0.5, 0.9]),
}).drop(index="count").round(0))

In [ ]:
# 2d Is the scope really English? The scope cut keeps a listing when the provider's `language` attribute says
# "en" (EDA_SDA_1_Lean 4c) and nothing reads the words, so a listing labelled "en" whose description is in another
# language survives (one turned up by hand in the 2b review). Scored here on the cleaned text 2b already wrote, so
# no cache is rebuilt and no new package is needed: each description's English function words are counted against
# five other languages', and whichever side wins decides. On synthetic job ads every English variant (prose, terse
# bullets, all-caps legal text, an English ad with a Spanish job title) scores 0.93 or more and every other language
# 0.18 or less, so the 0.5 cut has a wide margin either way. Cached: delete eda3_language.parquet to rebuild.
EN_WORDS = r"\b(?:the|and|of|to|for|with|you|we|our|are|is|be|will|that|this|have|from|as|or|an|a)\b"
OTHER_WORDS = (r"\b(?:de|la|el|los|las|del|y|que|para|con|un|una|por|su|sus|como|se|es|mas|o"       # Spanish
               r"|du|des|le|les|et|pour|avec|dans|est|une|sur|au|aux|vous|nous|ses"                 # French
               r"|und|der|die|das|den|dem|ein|eine|mit|fuer|für|sie|ist|im|zu|auf|von|bei|als"  # German
               r"|ng|sa|ang|mga|ay|na|ito"                                                          # Tagalog
               r"|e|em|do|da|dos|das|nao|não|voce|você)\b")                               # Portuguese
LANG_MIN_WORDS = 30  # under this a zero score is chance, not evidence: left unflagged, counted on its own line
EN_SHARE = 0.5       # flagged when other languages' function words outnumber English's
NON_LATIN = 0.10     # or when this much of the text is outside Latin-1 (Chinese, Japanese, Russian, Arabic)

if LANG_PATH.exists():
    lang = pd.read_parquet(LANG_PATH)
    print(f"loaded language scores for {len(lang):,} descriptions from {LANG_PATH.name} (delete it to rebuild 2d)")
else:
    scored = []
    for b in pq.ParquetFile(CLEAN_PATH).iter_batches(batch_size=50_000, columns=["id", "desc_clean"]):
        f = b.to_pandas()
        lower = f.desc_clean.str.lower()
        n_words = f.desc_clean.str.count(r"\S+")
        en_hits, other_hits = lower.str.count(EN_WORDS), lower.str.count(OTHER_WORDS)
        scored.append(pd.DataFrame({
            "id": f.id.to_numpy(),
            "lang_words": n_words.to_numpy(),
            "en_share": (en_hits / (en_hits + other_hits).clip(lower=1)).astype("float32").to_numpy(),
            "non_latin": (f.desc_clean.str.count(r"[^\x00-\xff]")
                          / f.desc_clean.str.len().clip(lower=1)).astype("float32").to_numpy(),
        }))
    lang = pd.concat(scored, ignore_index=True)
    enough = lang.lang_words >= LANG_MIN_WORDS
    lang["not_english"] = (enough & lang.en_share.lt(EN_SHARE)) | lang.non_latin.gt(NON_LATIN)
    lang.to_parquet(LANG_PATH, index=False)
    print(f"saved language scores for {len(lang):,} descriptions to {LANG_PATH.name}")
    del scored
# dropped first, so the cell can be re-run in a live kernel without suffixing the column
jobs = jobs.drop(columns="not_english", errors="ignore").merge(lang[["id", "not_english"]], on="id", how="left")
jobs["not_english"] = jobs.not_english.eq(True)  # no description means nothing to read, so nothing to flag
enough = lang.lang_words >= LANG_MIN_WORDS

show("2d English share of function words (descriptions of 30+ words)",
     lang.en_share[enough].describe(percentiles=[0.001, 0.01, 0.05, 0.5]).round(3).to_frame("value"))
counted = pd.DataFrame({"listings": {
    "not English (either test)": int(lang.not_english.sum()),
    "  other languages' function words win": int((enough & lang.en_share.lt(EN_SHARE)).sum()),
    f"  over {NON_LATIN:.0%} of characters outside Latin-1": int(lang.non_latin.gt(NON_LATIN).sum()),
    "borderline (0.5 to 0.7), worth reading by hand": int((enough & lang.en_share.between(EN_SHARE, 0.7)).sum()),
    f"under {LANG_MIN_WORDS} words, not scored": int((~enough).sum()),
}})
counted["share of the scope"] = (counted.listings / len(jobs)).round(5)
counted["share of listings with a description"] = (counted.listings / int(has_text.sum())).round(5)
show("2d What a text-based English check would drop", counted)

# The scope is label-English by construction, so this should report every flagged listing as labelled "en": the
# provider's label is wrong, rather than the scope cut having kept a listing it should have dropped.
labels = (pd.read_parquet(SCOPE_TABLE, columns=["id", "language"])
          if "language" in set(pq.ParquetFile(SCOPE_TABLE).schema.names) else None)
print("the provider's language label on the flagged listings:",
      labels.loc[labels.id.isin(set(lang.id[lang.not_english])), "language"].value_counts(dropna=False).to_dict()
      if labels is not None else "no language column in the Lean scope table")

show("2d Flagged share by role bucket", (jobs[has_text].groupby("role_bucket", observed=True)
     .not_english.agg(["sum", "mean"]).rename(columns={"sum": "listings flagged", "mean": "share of the bucket"})
     .sort_values("listings flagged", ascending=False).round(4)))


In [ ]:
# 2d review: a few flagged descriptions, for a human to confirm they really are not English, plus the two closest to
# the 0.5 cut, where a wrong call is likeliest. Shows real listing text. Tagged "data-rows": clear before committing.
hits = lang[lang.not_english]
if hits.empty:
    print("nothing flagged, so nothing to read")
else:
    pick = pd.concat([hits.sample(min(3, len(hits)), random_state=7),
                      hits.assign(margin=(hits.en_share - EN_SHARE).abs()).nsmallest(2, "margin")])
    pick = pick.drop_duplicates("id")
    text = pq.read_table(CLEAN_PATH, columns=["id", "desc_clean"],
                         filters=[("id", "in", pick.id.tolist())]).to_pandas()
    for row in pick.merge(text, on="id").itertuples():
        print("=" * 100, f"\nEnglish share of function words {row.en_share:.2f}, "
                         f"non-Latin {row.non_latin:.0%}, {row.lang_words:,} words\n", row.desc_clean[:1200])


In [ ]:
# 2 Chart: words per description at each cleanup step (left), and what boilerplate removal takes out (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
bins = np.logspace(0, np.log10(max(jobs.words_raw.max(), 10)), 40)
for vals, label, color in [(jobs.words_raw[has_text], "after HTML removal", MUTED), (jobs.words_clean[has_text], "after boilerplate removal", BLUE),
                           (words_lem, "lemmas, stopwords removed (sample)", ORANGE)]:
    ax1.hist(vals.clip(lower=1), bins=bins, histtype="step", lw=1.8, color=color, label=label, density=True)
ax1.set(xscale="log", title="Words per description at each step", xlabel="words (log scale)", ylabel="share of listings (density)")
ax1.legend(fontsize=8, frameon=False)
bar_h(ax2, 100 * by_reason / jobs.words_raw.sum(), "Share of all description words removed", "% of words", "{:.1f}%")
finish()

**What the data shows**

_(to write from this notebook's own run on the Lean chain. The original's numbers are gone because its scope was EDA_SDA_2's 908,291 listings and this one is EDA_SDA_1_Lean's 878,937, so not one of them carries across. This cell answers: what the chart shows, how words per description fall at each cleanup step, and whether the longest descriptions are long before and after.)_


## 3. The blocker rules: years of experience and degree

EDA_SDA_1_Lean's rules read every line the same way: "3+ years required" and "5 years preferred" both count, and its degree rule only fires when "required" comes within six words of "bachelor's", so the most common wording ("Bachelor's degree in Computer Science or a related field") never matched, and it fires on very few listings (3b measures both here). Here both are rebuilt on the cleaned text, with **requirement lines told apart from preferred ones** (from the headings kept in 2a, and lines that say "preferred" or "a plus"):

- **Years:** `years_asked` (any line, like EDA_SDA_1_Lean), `years_required` (requirement lines only), `years_preferred`. Also reads "minimum of two years" and number words.
- **Degree:** `degree_required` (a degree named on a requirement line), `degree_preferred`, `degree_equivalent` ("or equivalent experience", "in lieu of a degree"), `degree_level` (the highest level required).

The question for the app: **how many listings would each rule hide if a user turned it on** (too many years required; a degree they don't have). These are unvalidated keyword rules (decided 2026-10-13), so they label listings and drive opt-in filters; nothing hides by default.

In [ ]:
# 3a Years of experience: the new rules vs. EDA_SDA_1_Lean's, and how many listings each would hide at 5+ years
def share(mask):
    """Share of rows where mask is true; a blank (no years stated) counts as false, so every share is over all rows."""
    return mask.fillna(False).astype(bool).mean()

d = jobs[has_text]
y_old, y_new = d.years_required_eda1.astype(float), d.years_asked.astype(float)
agree = pd.Series({
    "both say no years": share(y_old.isna() & y_new.isna()),
    "same number": share(y_old == y_new),
    "new rule finds more years (or finds years where EDA_SDA_1_Lean found none)": share(y_new > y_old.fillna(0)),
    "new rule finds fewer (or none where EDA_SDA_1_Lean found some)": share(y_new.fillna(0) < y_old),
}, name="share of listings with a description")
show("3a EDA_SDA_1_Lean's years rule vs. the new one (any line)", agree.to_frame())
open_now = d.live
y_req, y_pref = d.years_required.astype(float), d.years_preferred.astype(float)
rules = {
    "EDA_SDA_1_Lean: 5+ years anywhere": (y_old >= 5),
    "new: 5+ years anywhere": (y_new >= 5),
    "new: 5+ years on a requirement line": (y_req >= 5),
    "new: 5+ years only on preferred lines": (y_pref >= 5) & ~(y_req >= 5),
}
show("3a The 5+ years rule: share of listings with a description, and of open ones (what it would hide)", pd.DataFrame({
    "all listings": {k: share(v) for k, v in rules.items()},
    "open on September 30": {k: share(v[open_now]) for k, v in rules.items()},
}).round(3))
by_bucket = d.assign(y_new=y_new, y_req=y_req, y_old=y_old).groupby("role_bucket").agg(
    states_years=("y_new", lambda s: s.notna().mean()),
    median_required=("y_req", "median"),
    required_5plus=("y_req", lambda s: (s >= 5).mean()),
    required_2_or_fewer=("y_req", lambda s: (s <= 2).mean()),
    eda1_5plus=("y_old", lambda s: (s >= 5).mean()),
    listings=("id", "size"),
).sort_values("required_5plus")
show("3a Years required by role bucket (listings with a description)", by_bucket.round(3))

In [ ]:
# 3b Degree: the new rule vs. EDA_SDA_1_Lean's, with "or equivalent experience" and the level asked
d = jobs[has_text]
show("3b Degree rules: share of listings with a description", pd.Series({
    "EDA_SDA_1_Lean: degree required": d.degree_required_eda1.mean(),
    "new: a degree mentioned anywhere": d.degree_mentioned.mean(),
    "new: degree on a requirement line": d.degree_required.mean(),
    "new: degree only on preferred lines": (d.degree_preferred & ~d.degree_required).mean(),
    "new: or equivalent experience accepted": d.degree_equivalent.mean(),
    "new: degree required, no equivalent (the per-applicant blocker)": (d.degree_required & ~d.degree_equivalent).mean(),
}, name="share").round(3).to_frame())
show("3b Highest degree level on a requirement line", d.degree_level.value_counts(normalize=True, dropna=False).rename("share").round(3).to_frame())
show("3b Degree required (no equivalent) by role bucket", d.groupby("role_bucket").apply(
    lambda g: pd.Series({"degree required": g.degree_required.mean(), "no equivalent": (g.degree_required & ~g.degree_equivalent).mean(),
                         "bachelor's or higher": g.degree_level.isin(["bachelor", "master", "phd"]).mean()}),
    include_groups=False).sort_values("no equivalent").round(3))

In [ ]:
# 3 Chart: share asking 5+ years on a requirement line vs. EDA_SDA_1_Lean's rule, by role bucket
fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(by_bucket))
ax.barh(y - 0.2, 100 * by_bucket.eda1_5plus, height=0.4, color=MUTED, label="EDA_SDA_1_Lean: 5+ years anywhere")
ax.barh(y + 0.2, 100 * by_bucket.required_5plus, height=0.4, color=BLUE, label="new: 5+ years on a requirement line")
ax.set(yticks=y, yticklabels=by_bucket.index, xlabel="% of listings with a description", title="Who asks 5+ years: old rule vs. requirement lines only")
ax.legend(fontsize=8, frameon=False)
finish()

**What the data shows**

_(to write from this notebook's own run on the Lean chain. The original's numbers are gone because its scope was EDA_SDA_2's 908,291 listings and this one is EDA_SDA_1_Lean's 878,937, so not one of them carries across. This cell answers: what the chart shows; how far EDA_SDA_1_Lean's years rule is too narrow; whether required and preferred matter for years; how the years asked change by role bucket; how far EDA_SDA_1_Lean's degree rule undercounts; which degree level is asked for most; where the degree blocker is lowest and highest; and the standing note that these are unvalidated keyword rules.)_


## 4. Other text features

Each feature is a keyword rule on the cleaned text (`src/text_features.py`), reported with its firing rate and not hand-validated (decided 2026-10-13); it labels listings and can be an opt-in filter, and hides nothing by default. For each: how often it fires, and whether it relates to how fast listings close **beyond the confounders** (the 7-day close rate's gap with vs. without the feature, within company volume band × source type cells, on original listings with a full week of follow-up; EDA_SDA_4 tests the survivors properly). Listings where a feature is unknown (NA) are left out of that feature's comparison, not counted as "without"; the `known` column in 4b shows each feature's coverage (1 = every listing in the comparison is known).

| Group | Features |
|:--|:--|
| Labels and opt-in filters for some applicants | `clearance_required` (vs. `clearance_obtainable`), `no_sponsorship`, `us_citizen_required`, `cert_required` and named certifications, `shift_or_oncall`, `travel_pct`, `contract_to_hire` |
| What a newcomer is offered | `entry_wording`, `changer_wording` (now also bootcamp graduates, self-taught) |
| Hiring difficulty and urgency | `sign_on_bonus`, `hiring_immediately`, `relocation_offered`, `urgency_context` ("urgent care" no longer counts) |
| Evergreen wording (ghost signal) | six phrase groups and `evergreen_any`, compared with EDA_SDA_2_Lean's likely-ghost flag |
| Remote and length | `remote_mention` (minus "remote patient monitoring" and the like), `words_clean` |

In [ ]:
# 4a How often each text feature fires, overall and by role bucket (listings with a description)
FLAGS = [c for c in tf.FLAG_COLS if c not in ("degree_mentioned", "degree_required", "degree_preferred", "degree_equivalent")]
d = jobs[has_text]
rates = d[FLAGS].astype(float).mean().sort_values(ascending=False)
show("4a Share of listings with a description where each feature fires", rates.rename("share").round(4).to_frame())
show_shaded("4a Text features by role bucket (share of listings; features firing on 0.5%+)",
            d.groupby("role_bucket")[[c for c in FLAGS if rates[c] >= 0.005]].mean().T.round(3))

In [ ]:
# 4b Does each feature go with faster or slower closing, beyond the confounders? Original listings with a full week of
# follow-up; the 7-day close rate with vs. without the feature, raw and within volume band × source type cells
# (weighted by the cell's listings). A first look: EDA_SDA_4's adjusted model decides.
cut = pd.Timestamp("2026-09-23", tz=jobs.first_seen.dt.tz)
o = jobs[has_text & ~jobs.is_repost & (jobs.first_seen <= cut)].copy()
o["closed_7d"] = (~o.live & (o.days_open <= 7)).astype(float)
o["cell"] = o.volume_band.astype(str) + " | " + o.source_type.astype(str)

def within_gap(flag):
    k = o[o[flag].notna()]  # unknown (NA) listings sit out this feature's comparison
    g = k.groupby(["cell", k[flag].astype(bool)]).closed_7d.agg(["mean", "size"]).unstack()
    g = g.dropna()
    g = g[(g[("size", True)] >= 200) & (g[("size", False)] >= 200)]
    if g.empty:
        return np.nan
    w = g[("size", True)] + g[("size", False)]
    return float(((g[("mean", True)] - g[("mean", False)]) * w).sum() / w.sum())

rows = {}
for f in FLAGS + ["degree_required", "degree_equivalent"]:
    k = o[o[f].notna()]  # unknown (NA) listings sit out, rather than counting as "without"
    on = k[f].astype(bool)
    if on.sum() < 500:
        continue
    rows[f] = {"share": on.mean(), "known": len(k) / len(o),
               "closed in 7 days (with)": k.closed_7d[on].mean(), "closed in 7 days (without)": k.closed_7d[~on].mean(),
               "raw gap (points)": 100 * (k.closed_7d[on].mean() - k.closed_7d[~on].mean()), "gap within confounders (points)": 100 * within_gap(f)}
show("4b Text features vs. closing within 7 days (original listings with a description and a full week of follow-up)",
     pd.DataFrame(rows).T.sort_values("gap within confounders (points)").round(3) if rows
     else pd.DataFrame({"note": ["no feature fires on 500+ of these listings"]}))

In [ ]:
# 4c Evergreen wording vs. EDA_SDA_2_Lean's ghost signals: does it add to "old and from a quiet company"?
d = jobs[has_text]
grp = pd.Series(np.where(d.likely_ghost, "likely ghost", np.where(d.stale, "open 90+ days, company still posting",
                np.where(d.live, "open, 90 days or younger", "closed"))), index=d.index)
show("4c Evergreen wording by ghost status (share of listings with a description)",
     d.groupby(grp)[tf.EVERGREEN + ["evergreen_any"]].mean().round(4))

**What the data shows**

_(to write from this notebook's own run on the Lean chain. The original's numbers are gone because its scope was EDA_SDA_2's 908,291 listings and this one is EDA_SDA_1_Lean's 878,937, so not one of them carries across. This cell answers: how often each feature fires; which role buckets differ and where; how big the 7-day close-rate gaps are; which raw gaps change sign once the confounders are held fixed; and whether evergreen wording lines up with EDA_SDA_2_Lean's likely-ghost flag.)_


## 5. Validation samples (not being marked)

**Decided 2026-10-13: these samples will not be hand-checked.** 5a still writes a **sample** for each rule: 60 listings where it fires and 60 where it doesn't, stratified by role bucket, to `data/clean/eda3_validation/<rule>.csv` (local only), and 5b would read marks back and report precision and recall if anyone ever fills them in. Nobody is marking them, so **the rules are reported as keyword rules with their firing rates (sections 3, 4 and 7), not as validated blockers.** That sets what each rule may do: the text blockers (years asked, degree required with no equivalent, clearance, no sponsorship, required certification and the other career-change flags) are **labels on each listing and opt-in filters a career changer turns on, never hidden by default**. Only reliable rules hide by default (seniority, likely ghosts, occupations that aren't a tech path, all decided in EDA_SDA_2). Every rule reports how many listings it would hide (7a). Listing text never leaves the laptop.

In [ ]:
# 5a Write the hand-check samples (only if they don't exist yet, so marks are never overwritten). Shows counts only.
# Not being marked (decided 2026-10-13): kept so the rules could still be checked later.
RULES = ["years_required_5plus", "degree_required", "degree_equivalent", "clearance_required", "no_sponsorship", "cert_required",
         "entry_wording", "changer_wording", "evergreen_any", "urgency_context"]
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
d = jobs[has_text].assign(years_required_5plus=jobs.years_required.astype(float) >= 5)
written = {}
for rule in RULES:
    path = VALIDATION_DIR / f"{rule}.csv"
    if path.exists():
        continue
    picks = []
    for value in (True, False):  # about 60 matches and 60 non-matches, spread across role buckets by their size
        pool = d[d[rule].fillna(False).astype(bool) == value]
        if len(pool):
            picks.append(pool.groupby("role_bucket").sample(frac=min(60 / len(pool), 1), random_state=42).assign(fires=value))
    sample = pd.concat(picks)
    sample.assign(rule=rule, correct="")[["id", "rule", "fires", "role_bucket", "correct"]].to_csv(path, index=False)
    written[rule] = len(sample)
show("5a Hand-check samples written (open each CSV locally with the listing text from eda3_desc_clean.parquet)",
     pd.Series(written, name="rows").to_frame() if written else pd.DataFrame({"note": ["all samples already exist"]}))

In [ ]:
# 5b Read the marks back: precision of each rule (and recall from the non-matches), if anyone ever fills them in
# (not planned: decided 2026-10-13, the rules stay unvalidated labels and opt-in filters)
res = {}
for path in sorted(VALIDATION_DIR.glob("*.csv")):
    v = pd.read_csv(path)
    v = v[v.correct.isin([0, 1, "0", "1"])].assign(correct=lambda x: x.correct.astype(int))
    if v.empty:
        continue
    hit, miss = v[v.fires], v[~v.fires]
    res[path.stem] = {"marked": len(v), "precision": hit.correct.mean() if len(hit) else np.nan,
                      "share of non-matches that are right": miss.correct.mean() if len(miss) else np.nan,
                      "hides listings (precision 90%+)": bool(len(hit) and hit.correct.mean() >= 0.9)}
show("5b Validation results", pd.DataFrame(res).T if res else pd.DataFrame({"note": ["no marks: the samples are not being marked (decided 2026-10-13), so the rules stay unvalidated"]}))

**What the data shows**
- **No rule is validated, and none will be:** the samples are written (about 120 rows per rule, for ten rules: 5+ years required, degree required, degree equivalent, clearance, no sponsorship, required certification, entry wording, career-changer wording, evergreen wording, urgency) but are not being marked (decided 2026-10-13), so 5b reports no precision or recall.
- **So the rules are keyword rules reported with their firing rates:** they are labels and opt-in filters, never hidden by default; the 90% precision bar that would have let a rule hide listings is not applied to any of them.

## 6. Skills mapped to O*NET

**Question:** which tools and skills does each listing ask for, and how do they line up with what O*NET says the occupation needs? The answer feeds `ML_Fit`'s skills-based matching and the retraining compass's "skills that carry over".

Data: **O*NET 31.0 Database, USDOL/ETA (CC BY 4.0)**, downloaded with `python -m fetch.get_onet` into `data/external/onet/`. Release 31.0 splits skills into Essential Skills, Transferable Skills and Software Skills; `src/onet.py` reads them (importance `IM` and level `LV` ratings only; suppressed and not-relevant ratings dropped).

| Step | What | Output |
|:--|:--|:--|
| 6a | **Skill vocabulary:** every software tool O*NET lists (the Workplace Example, with its hot-technology and in-demand flags) plus the skill element names, lowercased with punctuation removed and common aliases folded together ("js" to "javascript", "MS Excel" to "Microsoft Excel"; C++, C# and .NET keep their meaning). Matched as **whole phrases**, so "java" is not found inside "javascript". Everyday words that are also tool names ("Act!", "Analyze", "Google" alone, "Tanks", "Reduce", "Sand", "Freight+", "Route", "Dig", "Word", "Access", ...) and the generic one-word skill names (Science, Writing, Speaking, Mathematics, Monitoring, Coordination, Persuasion, Instructing, Installation, Repairing) are left out, because matching them as phrases fires on ordinary sentences; "R", "Go", "C" and "Swift" are matched only in context | The vocabulary and a compiled matcher |
| 6b | **Extraction from requirement lines only** (a line under a requirements or preferred heading, or with a requirement cue such as "years", "experience", "proficiency"; responsibilities are not read), batch by batch from `eda3_desc_clean.parquet` with the same crash-safe `.part` writes as 2a-2b. Each match is marked **required** (on any required line) or **preferred** | `eda3_skills.parquet`: `id`, `name`, `kind` (tool or skill), `section`, `element_id` (cached; delete to rebuild) |
| 6c | **Coverage** and the most-asked tools and skills per role bucket | Aggregates |
| 6d | **Agreement with the listing's own occupation:** the share of a listing's matched skills that O*NET lists for its O*NET-SOC code (tools on O*NET's Software Skills list for the code; skills rated 3+ of 5 in importance, Essential or Transferable) | Aggregates |
| 6e | **Job Zone per bucket** (O*NET's 1-5 scale of preparation needed) as an accessibility prior | Aggregates |
| 6f | **Transferable skills that carry over** from each of Profiles' prior-career areas (`BACKGROUND_AREAS`, as O*NET groups in `BACKGROUND_SOC`) to each target bucket: the compass's "skills that carry over"; in the app, a user's own résumé is matched the same way (`ML_Fit`) | Aggregates |

Limits: a skill element name such as "Programming" or "Time Management" can still be ordinary wording, so skill matches are looser than tool matches; the keyword rules are not hand-checked (as in section 4), so they are labels and matching features, never filters that hide listings.

In [ ]:
# 6a Load the O*NET tables (data/external/onet/ from fetch.get_onet) and build the skill vocabulary and matcher
from src import onet
SKILLS_PATH = LEAN_DIR / "eda3_skills.parquet"  # id, name, kind, section, element_id (cache; delete to rebuild)
essential, transferable = onet.ratings("Essential Skills"), onet.ratings("Transferable Skills")
software_tbl, zones = onet.software(), onet.job_zones()
vocab = onet.vocabulary(software_tbl, [essential, transferable])
matcher = onet.Matcher(vocab)
show("6a Skill vocabulary (O*NET 31.0 Database, USDOL/ETA, CC BY 4.0)", pd.DataFrame({"value": {
    "O*NET occupations with essential-skill ratings": f"{essential.code.nunique():,}",
    "software tools (Workplace Examples) kept": f"{(vocab.kind == 'tool').sum():,}",
    "  of them hot technologies": f"{vocab.hot[vocab.kind == 'tool'].sum():,}",
    "  of them in demand": f"{vocab.in_demand[vocab.kind == 'tool'].sum():,}",
    "skill element names kept": f"{(vocab.kind == 'skill').sum():,}",
    "names left out (very short or everyday words)": f"{vocab.attrs['skipped']:,}",
    "names matched only in context (R, Go, C, Swift)": f"{int(vocab.context.sum()):,}",
}}))

In [ ]:
# 6b Find each listing's tools and skills in its requirement lines, batch by batch on several processes (cached: delete
# eda3_skills.parquet to rebuild; the cache is rebuilt if the O*NET vocabulary or the cleaned text changed)
if onet.cache_valid(SKILLS_PATH, CLEAN_PATH, matcher):
    print(f"loaded the skills from {SKILLS_PATH.name} (delete it to rebuild 6b)")
else:
    start = time.time()
    n_rows = onet.extract(CLEAN_PATH, matcher, SKILLS_PATH,
                          progress=lambda n, k: print(f"  batch {n + 1}: {k:,} skill rows, {time.time() - start:.0f}s", end="\r"))
    print(f"\nfound {n_rows:,} skill rows in {(time.time() - start) / 60:.1f} minutes")
sk = pd.read_parquet(SKILLS_PATH, columns=["id", "name", "kind", "section"])
for c in ["name", "kind", "section"]:
    sk[c] = sk[c].astype("category")
sk = sk.merge(jobs[["id", "role_bucket"]], on="id", how="left")
print(f"{len(sk):,} skill rows over {sk.id.nunique():,} listings; {(sk.section == 'required').mean():.0%} of rows are on required lines")

In [ ]:
# 6c Coverage (share of listings with a description that have at least one matched skill) and the most-asked
# tools and skills per role bucket
d = jobs[has_text]
cov = d.assign(any_skill=d.id.isin(sk.id), required_skill=d.id.isin(sk.id[sk.section == "required"]),
               any_tool=d.id.isin(sk.id[sk.kind == "tool"])).groupby("role_bucket")[["any_skill", "required_skill", "any_tool"]].mean()
cov.loc["all"] = d.id.isin(sk.id).mean(), d.id.isin(sk.id[sk.section == "required"]).mean(), d.id.isin(sk.id[sk.kind == "tool"]).mean()
per_listing = sk.groupby("id", observed=True).size()
show("6c Coverage: share of listings with a description (by role bucket)", cov.rename(columns={
    "any_skill": "at least one skill", "required_skill": "one on a required line", "any_tool": "at least one tool"}).round(3))
print(f"matched skills per listing that has any: median {per_listing.median():.0f}, 90th percentile {per_listing.quantile(0.9):.0f}")
n_bucket = d.groupby("role_bucket").size()
for kind, k in [("tool", 20), ("skill", 15)]:
    counts = sk[sk.kind == kind].groupby(["name", "role_bucket"], observed=True).id.nunique().unstack(fill_value=0)
    counts = counts.reindex(columns=n_bucket.index, fill_value=0)
    top = counts.sum(axis=1).nlargest(k).index
    show_shaded(f"6c Most-asked {kind}s: share of each bucket's listings that ask for it (top {k} overall)", (counts.loc[top] / n_bucket).round(3))
hot = vocab[vocab.kind == "tool"].set_index("name")[["hot", "in_demand"]]
t = sk[sk.kind == "tool"]
show("6c Share of tool mentions that are O*NET hot technologies or in demand", pd.Series({
    "hot technology": t.name.map(hot.hot).astype(float).mean(), "in demand": t.name.map(hot.in_demand).astype(float).mean()}, name="share").round(3))

In [ ]:
# 6c Chart: coverage by role bucket
fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(cov))
ax.barh(y - 0.2, 100 * cov.any_skill, height=0.4, color=MUTED, label="at least one skill or tool")
ax.barh(y + 0.2, 100 * cov.required_skill, height=0.4, color=BLUE, label="one on a required line")
ax.set(yticks=y, yticklabels=cov.index, xlabel="% of listings with a description", title="Listings with a matched O*NET skill or tool, by role bucket")
ax.legend(fontsize=8, frameon=False)
finish()

In [ ]:
# 6d Agreement with the listing's own occupation: of the skills matched in a listing, the share O*NET lists for its
# O*NET-SOC code (tools from Software Skills; skills O*NET rates 3+ of 5 in importance, Essential or Transferable)
HAVE_CODE = "onet_code" in jobs.columns
if not HAVE_CODE:
    print("onet_code is not in EDA_SDA_1_Lean's in-scope table: save it there to run 6d-6f")
else:
    codes = jobs.set_index("id").onet_code
    listed = onet.listed_by_code(codes.dropna().unique(), software_tbl, [essential, transferable])  # skills rated 3+ of 5 for the code
    rows = {}
    for label, part in [("tools and skills", sk), ("tools only", sk[sk.kind == "tool"]), ("skills only", sk[sk.kind == "skill"])]:
        g = onet.agreement(part, codes, listed).join(jobs.set_index("id").role_bucket)
        rows[label] = g.groupby("role_bucket").share.median()
        rows[label].loc["all"] = g.share.median()
    show("6d Median share of a listing's matched skills that O*NET lists for its own code (by role bucket)", pd.DataFrame(rows).round(3))
    g_all = onet.agreement(sk, codes, listed)
    print(f"{len(g_all):,} listings have a code and a match; {(g_all.n_listed == 0).mean():.1%} have none of their matches on O*NET's list for their code")

In [ ]:
# 6e Job Zone per role bucket (1 little preparation needed ... 5 extensive) as an accessibility prior
if HAVE_CODE:
    jz = onet.job_zone_for(codes.reindex(jobs.id).reset_index(drop=True), zones)
    jz.index = jobs.index
    by_zone = pd.crosstab(jobs.role_bucket, jz.round().astype("Int64"), normalize="index")
    by_zone.columns = [f"zone {c}" for c in by_zone.columns]
    by_zone["mean zone"] = jz.groupby(jobs.role_bucket).mean()
    show_shaded("6e Job Zone per role bucket (share of listings; last column the mean)", by_zone.round(3))
    print(f"{jz.isna().mean():.1%} of listings have no Job Zone (no O*NET code, or a code O*NET does not list)")

In [ ]:
# 6f Skills that carry over: transferable skills (importance 3+ of 5) that a background area and a target bucket both
# need. Backgrounds are Profiles' prior-career areas (BACKGROUND_AREAS), as O*NET-SOC groups (BACKGROUND_SOC in
# src/career_switch.py); targets are the scope's buckets, weighted by their listings. Public data only.
from src.career_switch import BACKGROUND_SOC
if HAVE_CODE:
    matrix = onet.skill_matrix(transferable)
    code10 = jobs.assign(code=codes.reindex(jobs.id).to_numpy()).dropna(subset=["code"])
    weights = code10.groupby(["role_bucket", "code"]).size()
    targets = {b: onet.profile(matrix, list(w.droplevel(0).index), dict(w.droplevel(0))) for b, w in weights.groupby(level=0)}
    show("6f Each bucket's most important transferable skills (mean importance 1-5, weighted by its listings)",
         pd.DataFrame({b: pd.Series([f"{k} ({v:.2f})" for k, v in p.dropna().nlargest(6).items()]) for b, p in targets.items()}).fillna(""))
    shared = {area: {b: onet.carry_over(onet.profile(matrix, prefixes), p) for b, p in targets.items()}
              for area, prefixes in BACKGROUND_SOC.items()}
    show_shaded("6f Transferable skills both rate 3+ of 5: Profiles background area (rows) by target bucket (columns)",
                pd.DataFrame({b: {a: len(s[b]) for a, s in shared.items()} for b in targets}))
    show("6f The three strongest shared skills per background area and bucket", pd.DataFrame(
        {b: {a: ", ".join(s[b].index[:3]) for a, s in shared.items()} for b in targets}))


**What the data shows** (O*NET 31.0 Database, USDOL/ETA, CC BY 4.0; keyword matching on requirement lines)

_(to write from this notebook's own run on the Lean chain. The original's numbers are gone because its scope was EDA_SDA_2's 908,291 listings and this one is EDA_SDA_1_Lean's 878,937, so not one of them carries across. This cell answers: skill coverage; the most-asked tools and skills; agreement with the listing's own occupation; Job Zone by bucket; which skills carry over from a prior career area; and the limits of keyword matching.)_


## 7. Final table

One row per scope listing (`id`), the text features from sections 3–4, and **text-rule flags** for the app (the `blocker_*` columns). They are unvalidated keyword rules (decided 2026-10-13): the app shows them as labels and opt-in filters (for example "I have no degree"), never hides by default, and 7a reports how many listings each would hide. Blank means unknown: the listing has no description. Saved as `text_table.parquet` for EDA_SDA_4 (joined by `id`) and `ML_Fit`.

In [ ]:
# 7a Text blockers: labels and opt-in filters in the app, never hidden by default (unvalidated keyword rules,
# decided 2026-10-13), and what each would hide among open listings if a user turned it on
out = jobs[["id"] + tf.FEATURE_COLS].copy()
known = jobs.words_raw.fillna(0) > 0
out["blocker_years_5plus"] = (jobs.years_required.astype(float) >= 5).where(known)  # a filter anyone can turn on (no years stated = False)
out["blocker_degree_no_equivalent"] = (jobs.degree_required & ~jobs.degree_equivalent).where(known)  # per applicant
out["blocker_clearance"] = jobs.clearance_required.where(known)                 # per applicant
out["blocker_no_sponsorship"] = jobs.no_sponsorship.where(known)                # per applicant (needs sponsorship)
out["blocker_cert_required"] = jobs.cert_required.where(known)                  # per applicant
BLOCKERS = [c for c in out if c.startswith("blocker_")]
show("7a Candidate blockers: share of listings with a description, and of open ones", pd.DataFrame({
    "all listings": out.loc[known, BLOCKERS].astype(float).mean(),
    "open on September 30": out.loc[known & jobs.live, BLOCKERS].astype(float).mean(),
}).round(3))
out.to_parquet(TEXT_TABLE, index=False)
show("7a The final text table", pd.DataFrame({"value": {
    "listings": f"{len(out):,}", "columns": out.shape[1], "with text features": f"{known.mean():.1%}",
    "saved": f"{TEXT_TABLE.name} ({TEXT_TABLE.stat().st_size / 1e9:.2f} GB)",
}}))
print(f"[EDA_SDA_3_Lean done in {(time.time() - _t0) / 60:.1f} min]")

## 8. Conclusion

_(to write from this notebook's own run on the Lean chain. Everything below was measured on EDA_SDA_2's 908,291
listings; this notebook runs on EDA_SDA_1_Lean's 878,937, so every number changes and none is carried over. The
headings and the questions stay, because those are the notebook's own shape.)_

**Summary** - one bullet each for: the cleanup (listings, share with a description, the median description before
and after boilerplate removal, the largest kind removed); the rebuilt rules (5+ years on a requirement line
against EDA_SDA_1_Lean's rule, degree required and how many accept no equivalent); the other text features (which
fire, how rarely, and how big their 7-day close gaps are within the confounders); evergreen wording against
EDA_SDA_2_Lean's likely-ghost flag; and the O\*NET skills (coverage, the most-asked tool and skill per bucket,
agreement with the listing's own code, Job Zone, and the skills that carry over).

**Answers to the clarifying questions**
- *Length: intent or boilerplate?* Not answered here; length (`words_clean`) is saved for `EDA_SDA_4`.
- *Urgency in context vs. close time?* _(firing rate and the gap within confounders, from 4a and 4b)_
- *Evergreen wording as a ghost signal?* _(from 4c, against EDA_SDA_2_Lean's `likely_ghost` and `open_90_days`)_
- *Are EDA_SDA_1_Lean's flags accurate?* _(from 3a and 3b; precision stays untested - the hand-check samples will
  not be marked, decided 2026-10-13)_
- *Which requirements hide a listing?* None by default. Years asked, clearance, no sponsorship, a required
  certification and a degree with no equivalent are **labels and opt-in filters**, each reported with how much it
  would hide if a user turned it on. **None is validated, so none hides by default.**
- *Skills mapped to O\*NET?* Yes, by keyword matching on requirement lines (section 6), saved as
  `eda3_skills.parquet`. Everyday-word tool names and generic one-word skill names are excluded; the matches are
  unchecked keyword hits, which a check should look at before they feed matching.

**Limits:** the gaps in 4b are a first look and associational, not causal; the rules are unvalidated keyword rules
(firing rates, no precision); the O\*NET skills are keyword matches (looser for skill names than for tools; R, Go,
C and Swift only in context; occupation-level, not employer-level); and the length question is open.

**Handoff** (all in `data/clean/lean/`, the Lean chain's folder): `text_table.parquet` (`id` + the text features
and the text-rule labels, blank where there is no description) for `EDA_SDA_4` and `ML_Fit`, plus
`eda3_desc_clean.parquet` (cleaned text), `eda3_lemmas_sample.parquet` and `eda3_skills.parquet` (`id`, `name`,
`kind`, `section`, `element_id`) for `ML_Fit`.
